# PeriPy Fracture Simulation Demo

A minimal, self-contained peridynamics fracture simulation built on top of the
[PeriPy](https://github.com/alan-turing-institute/PeriPy) library
(Alan Turing Institute, MIT licensed).

**Problem:** A rectangular 2-D plate with a horizontal pre-notch on the left edge
is pulled in tension (mode-I loading) until the crack propagates across the
specimen.

**What this notebook does**
1. Generates a simple triangular mesh of the plate on-the-fly (no external files needed).
2. Writes it as a Gmsh 2.2 `.msh` file, which is the format PeriPy consumes.
3. Defines a pre-crack, displacement boundary conditions, and material properties.
4. Runs a bond-based peridynamic simulation with PeriPy's OpenCL Euler integrator.
5. Visualises the resulting damage field and deformed shape with matplotlib.

**Why peridynamics?** Unlike classical continuum mechanics, peridynamics uses
non-local integral equations, so discontinuities like cracks can emerge and
propagate naturally — no special crack-tip enrichment required.

## 1. Environment setup

PeriPy depends on `pyopencl` and `cython`. On Windows, installing from source
is the most reliable path (a missing `.pyx` file in the PyPI wheel can break
`pip install peripy` — see PeriPy issue #127).

```bash
pip install numpy scipy matplotlib meshio h5py tqdm scikit-learn cython pyopencl
git clone https://github.com/alan-turing-institute/PeriPy.git
cd PeriPy && pip install -e .
```

You also need an OpenCL runtime installed for your CPU or GPU
(e.g. Intel OpenCL Runtime, POCL, or a GPU vendor driver).

In [ ]:
import sys
import pathlib

import numpy as np
import matplotlib.pyplot as plt

try:
    import meshio
    from peripy.model import Model, initial_crack_helper
    from peripy.integrators import EulerCL
    print('PeriPy is available.')
except ImportError as exc:
    print('Missing dependency:', exc)
    print('See the setup cell above for installation instructions.')
    raise

## 2. Geometry and mesh

We discretise a `L_x = 1.0 m` by `L_y = 0.5 m` plate into a regular triangle
mesh. Each quad in a structured grid is split into two triangles so PeriPy
can compute cell volumes correctly.

| Parameter | Value |
|---|---|
| Plate dimensions | 1.0 m × 0.5 m |
| Nodes along x    | 81 |
| Nodes along y    | 41 |
| Node spacing dx  | 0.0125 m |

We write the mesh as a Gmsh 2.2 `.msh` file so that PeriPy's loader can read it.

In [ ]:
L_x, L_y = 1.0, 0.5
nx, ny = 81, 41
dx = L_x / (nx - 1)
dy = L_y / (ny - 1)

# Build a regular grid of points in the z = 0 plane (meshio needs 3-D coords).
xs = np.linspace(0.0, L_x, nx)
ys = np.linspace(0.0, L_y, ny)
xx, yy = np.meshgrid(xs, ys)
points = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(nx * ny)])

# Build triangles: split each quad into two.
tris = []
for j in range(ny - 1):
    for i in range(nx - 1):
        n00 = j * nx + i
        n10 = j * nx + i + 1
        n01 = (j + 1) * nx + i
        n11 = (j + 1) * nx + i + 1
        tris.append([n00, n10, n11])
        tris.append([n00, n11, n01])
tris = np.asarray(tris, dtype=np.int64)

mesh_path = pathlib.Path('plate.msh')
meshio.write_points_cells(
    str(mesh_path),
    points=points,
    cells=[('triangle', tris)],
    file_format='gmsh22',
    binary=False,
)
print(f'Wrote {mesh_path.resolve()} with {len(points)} nodes and {len(tris)} triangles.')

## 3. Material properties and peridynamic parameters

We model a brittle, glass-like material with the classical bond-based
micro-modulus formula for 2-D:

$$c = \frac{9 E}{\pi t \, \delta^{3}}$$

where `E` is Young's modulus, `t` is plate thickness and `δ` is the horizon.
A bond breaks irreversibly when its stretch exceeds `s_0` (the critical stretch).

In [ ]:
# Material: brittle, glass-like
E = 72.0e9          # Pa   (Young's modulus)
rho = 2440.0        # kg/m^3 (density)
thickness = 0.01    # m    (plate thickness, 2-D bond-based formula)

# Peridynamic discretisation parameters
horizon = 3.015 * dx                          # typical m-value ~ 3
critical_stretch = 5.0e-4                     # bond failure strain
bond_stiffness = (9.0 * E) / (np.pi * thickness * horizon**3)

print(f'dx               = {dx:.4e} m')
print(f'horizon (delta)  = {horizon:.4e} m')
print(f'bond_stiffness   = {bond_stiffness:.4e} N/m^5')
print(f'critical_stretch = {critical_stretch:.2e}')

## 4. Pre-crack and boundary conditions

We introduce a horizontal notch on the mid-height of the left edge, extending
20 % of the plate length. Any bond that crosses the notch line is removed at
`t = 0`, which is done via PeriPy's `initial_crack_helper`.

The top and bottom edges are pulled outward with prescribed displacement
(mode-I loading).

In [ ]:
NOTCH_Y = L_y / 2.0
NOTCH_LENGTH = 0.2 * L_x

@initial_crack_helper
def is_crack(icoord, jcoord):
    """Return True if the bond between i and j crosses the notch segment."""
    # Bond must straddle the notch line in y ...
    if (icoord[1] - NOTCH_Y) * (jcoord[1] - NOTCH_Y) > 0.0:
        return False
    # ... and lie within the notch extent in x.
    x_mid = 0.5 * (icoord[0] + jcoord[0])
    return x_mid <= NOTCH_LENGTH

BC_LAYER = 2.0 * dx  # thickness of prescribed-displacement layers

def is_displacement_boundary(x):
    """Top edge pulled in +y, bottom edge pulled in -y.
    Returns a 3-element list: None = free, 1 / -1 = prescribed direction."""
    bnd = [None, None, None]
    if x[1] > L_y - BC_LAYER:
        bnd[1] = 1
    elif x[1] < BC_LAYER:
        bnd[1] = -1
    return bnd

## 5. Build the PeriPy model

We use the OpenCL Euler integrator (`EulerCL`). Time-step is chosen well
below the critical stable step for bond-based PD.

In [ ]:
dt = 1.0e-8  # s
integrator = EulerCL(dt=dt)

model = Model(
    mesh_file=str(mesh_path),
    integrator=integrator,
    horizon=horizon,
    critical_stretch=critical_stretch,
    bond_stiffness=bond_stiffness,
    dimensions=2,
    density=rho,
    initial_crack=is_crack,
    is_displacement_boundary=is_displacement_boundary,
)

print(f'Nodes        : {model.nnodes}')
print(f'Max neighbours per node: {model.max_neighbours}')
print(f'Total bonds  : {int(np.sum(model.family))} (2x counted)')

## 6. Run the simulation

We ramp the prescribed boundary velocity linearly from 0 to `v_max` and run
for `n_steps` time-steps. The simulation returns arrays for displacement,
velocity, acceleration, force, body force, damage and the neighbour list.

In [ ]:
n_steps = 2000
v_max = 2.0               # m/s boundary pull rate (arbitrary for demo)

# Prescribed displacement increment per step, ramped linearly.
displacement_bc_array = v_max * dt * np.ones(n_steps)

u, ud, udd, force, body_force, damage, nlist = model.simulate(
    displacement_bc_array=displacement_bc_array,
    write=200,
)
print('Simulation complete.')
print(f'  displacement shape : {u.shape}')
print(f'  damage shape       : {damage.shape}')

## 7. Visualisation

We plot the final damage field (per-node fraction of broken bonds) and the
deformed configuration, coloured by vertical displacement.

In [ ]:
coords = model.coords[:, :2]
disp_final = u  # PeriPy returns final-state arrays by default
damage_final = damage

# Deformed configuration (scaled for visibility)
scale = 50.0
deformed = coords + scale * disp_final[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sc0 = axes[0].scatter(
    coords[:, 0], coords[:, 1],
    c=damage_final, cmap='inferno', s=6, vmin=0.0, vmax=1.0,
)
axes[0].set_title('Damage field (undeformed)')
axes[0].set_xlabel('x [m]')
axes[0].set_ylabel('y [m]')
axes[0].set_aspect('equal')
plt.colorbar(sc0, ax=axes[0], label='damage')

sc1 = axes[1].scatter(
    deformed[:, 0], deformed[:, 1],
    c=disp_final[:, 1], cmap='coolwarm', s=6,
)
axes[1].set_title(f'Deformed shape (x{scale:.0f}), coloured by u_y')
axes[1].set_xlabel('x [m]')
axes[1].set_ylabel('y [m]')
axes[1].set_aspect('equal')
plt.colorbar(sc1, ax=axes[1], label='u_y [m]')

plt.tight_layout()
plt.show()

## 8. Next steps

Things you can try by tweaking this notebook:

- **Material**: swap in concrete, PMMA, or a metal by editing `E`, `rho` and
  `critical_stretch`.
- **Loading rate**: lower `v_max` for a quasi-static fracture, or raise it to
  study dynamic branching.
- **Notch geometry**: change `NOTCH_LENGTH` and `NOTCH_Y`, or add a second
  notch to study crack interaction.
- **Refinement**: bump `nx`, `ny` and reduce `dt` accordingly (`dt` must
  stay below the PD CFL limit).
- **Integrator**: try `VelocityVerletCL` for better energy conservation.

For production runs, PeriPy can write VTK / Exodus output through
`model.write_mesh(...)`, which you can open in ParaView.